- Instead of hooking onto before or after the agent, in this task. we hook into the model instance itself. This is not before or after the model, it is the model.
- This involves wrapping the model with custom middleware functions, also known as wrap-style middleware.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    context_schema=LanguageContext,
    middleware=[user_language_prompt] # The middleware allows switching prompts based on language context
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content[0]['text'])

Dia duit! Táim go maith, go raibh maith agat. Conas atá tú féin?


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content[0]['text'])

¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content[0]['text'])

Bonjour ! Je vais très bien, merci. Et vous, comment allez-vous aujourd'hui ?
